In [10]:
import numpy as np

token_ids = [4, 7, 2, 9, 5, 3, 8, 6, 1, 0]

print("\nBPE Output Token IDs")
print(token_ids)

vocab_size = max(token_ids) + 1

print("\nVocabulary Size")
print(vocab_size)

inputs = np.array(token_ids[:-1])

targets = np.array(token_ids[1:])

print("\nInput Tokens")
print(inputs)

print("\nTarget Tokens")
print(targets)

print("\nInput Shape")
print(inputs.shape)

print("\nTarget Shape")
print(targets.shape)

embedding_dimension = 8

embedding_matrix = np.random.randn(
    vocab_size,
    embedding_dimension
)

print("\nEmbedding Matrix Shape")
print(embedding_matrix.shape)

embedded_inputs = []

for token in inputs:

    vector = embedding_matrix[token]

    embedded_inputs.append(vector)

embedded_inputs = np.array(embedded_inputs)

print("\nEmbedded Input Shape")
print(embedded_inputs.shape)

print("\nFirst Embedded Vector")
print(embedded_inputs[0])

sequence_length = len(inputs)

position_encoding = np.zeros(
    (
        sequence_length,
        embedding_dimension
    )
)

for position in range(sequence_length):

    for dimension in range(embedding_dimension):

        angle = position / np.power(
            10000,
            (2 * (dimension // 2)) / embedding_dimension
        )

        if dimension % 2 == 0:

            position_encoding[position][dimension] = np.sin(angle)

        else:

            position_encoding[position][dimension] = np.cos(angle)

print("\nPositional Encoding Shape")
print(position_encoding.shape)

embedded_inputs = embedded_inputs + position_encoding

print("\nFinal Embedded Input Shape")
print(embedded_inputs.shape)

print("\nFirst Position Encoded Vector")
print(embedded_inputs[0])




BPE Output Token IDs
[4, 7, 2, 9, 5, 3, 8, 6, 1, 0]

Vocabulary Size
10

Input Tokens
[4 7 2 9 5 3 8 6 1]

Target Tokens
[7 2 9 5 3 8 6 1 0]

Input Shape
(9,)

Target Shape
(9,)

Embedding Matrix Shape
(10, 8)

Embedded Input Shape
(9, 8)

First Embedded Vector
[ 0.66722333 -0.6376137   0.70518192 -0.91414039 -0.39121028 -0.91654083
  1.87583104  0.65798901]

Positional Encoding Shape
(9, 8)

Final Embedded Input Shape
(9, 8)

First Position Encoded Vector
[ 0.66722333  0.3623863   0.70518192  0.08585961 -0.39121028  0.08345917
  1.87583104  1.65798901]


In [8]:

Wq = np.random.randn(embedding_dimension, embedding_dimension)
Wk = np.random.randn(embedding_dimension, embedding_dimension)
Wv = np.random.randn(embedding_dimension, embedding_dimension)

Query = embedded_inputs.dot(Wq)
Key = embedded_inputs.dot(Wk)
Value = embedded_inputs.dot(Wv)

print("\nQuery Shape :", Query.shape)
print("Key Shape   :", Key.shape)
print("Value Shape :", Value.shape)

scores = np.matmul(Query, Key.T)

scores = scores / np.sqrt(embedding_dimension)

print("\nAttention Scores Shape :")

print(scores.shape)


mask = np.triu(
    np.ones((sequence_length, sequence_length)),
    1
)

scores[mask == 1] = -np.inf

print("\nFuture Tokens Masked")

scores = scores - np.max(scores, axis=1, keepdims=True)

attention = np.exp(scores)

attention = attention / np.sum(
    attention,
    axis=1,
    keepdims=True
)

print("\nAttention Shape :")

print(attention.shape)

context = np.matmul(attention, Value)

print("\nContext Shape :")

print(context.shape)

Wo = np.random.randn(
    embedding_dimension,
    vocab_size
)

bias = np.random.randn(vocab_size)

logits = np.matmul(
    context,
    Wo
)

logits = logits + bias

print("\nLogits Shape :")

print(logits.shape)

logits = logits - np.max(
    logits,
    axis=1,
    keepdims=True
)

probabilities = np.exp(logits)

probabilities = probabilities / np.sum(
    probabilities,
    axis=1,
    keepdims=True
)

print("\nProbability Shape :")

print(probabilities.shape)

predicted = np.argmax(
    probabilities,
    axis=1
)

print("\nPredicted Token IDs")

print(predicted)

if 'id_to_token' in globals():

    print("\nFirst Predicted Token")

    print(id_to_token[predicted[0]])



Query Shape : (9, 8)
Key Shape   : (9, 8)
Value Shape : (9, 8)

Attention Scores Shape :
(9, 9)

Future Tokens Masked

Attention Shape :
(9, 9)

Context Shape :
(9, 8)

Logits Shape :
(9, 10)

Probability Shape :
(9, 10)

Predicted Token IDs
[4 4 4 6 5 9 5 4 4]


In [9]:

loss = 0

for i in range(len(targets)):

    loss += -np.log(probabilities[i][targets[i]] + 1e-9)

loss = loss / len(targets)

print("\nCross Entropy Loss :")

print(loss)

correct = np.sum(predicted == targets)

accuracy = (correct / len(targets)) * 100

print("\nAccuracy :")

print(round(accuracy, 2), "%")

gradient = probabilities.copy()

for i in range(len(targets)):

    gradient[i][targets[i]] -= 1

gradient = gradient / len(targets)

print("\nGradient Shape :")

print(gradient.shape)

learning_rate = 0.01

dWo = np.matmul(context.T, gradient)

dbias = np.sum(gradient, axis=0)

Wo = Wo - learning_rate * dWo

bias = bias - learning_rate * dbias

print("\nWeights Updated Successfully")

last_context = context[-1]

next_logits = np.matmul(last_context, Wo) + bias

next_logits = next_logits - np.max(next_logits)

next_probability = np.exp(next_logits)

next_probability = next_probability / np.sum(next_probability)

next_token = np.argmax(next_probability)

print("\nGenerated Next Token ID :")

print(next_token)

generated_sequence = list(token_ids)

generated_sequence.append(next_token)

print("\nGenerated Token Sequence :")

print(generated_sequence)

print("\n=========================================")
print("AUTOREGRESSIVE LANGUAGE MODEL COMPLETED")
print("=========================================")

print("\nProgram Completed Successfully")


Cross Entropy Loss :
14.263858133728844

Accuracy :
0.0 %

Gradient Shape :
(9, 10)

Weights Updated Successfully

Generated Next Token ID :
4

Generated Token Sequence :
[4, 7, 2, 9, 5, 3, 8, 6, 1, 0, np.int32(4)]

AUTOREGRESSIVE LANGUAGE MODEL COMPLETED

Program Completed Successfully
